# UAV Object Detection · Detectron2 lab
## 00 · Synthetic drone dataset
Every other notebook is fed by the generator written below. It renders **top-down (nadir) drone scenes** — parking lots and crop fields with **cars, people and trees** — and emits **COCO-format** annotations (boxes **and** polygon masks). It is pure `numpy`/`Pillow`, so it needs no GPU and no Detectron2 to produce data.

This mirrors the *drone imagery* used in the HTML study playgrounds; the deliberately tiny people reproduce the UAV **small-object** challenge.

In [ ]:
%%writefile drone_synth.py
"""
drone_synth.py — contextualized synthetic drone (nadir/aerial) imagery.

Generates top-down scenes (parking lots and crop fields) populated with
cars, people and trees, plus COCO-format annotations (bbox + polygon
segmentation). Pure numpy + Pillow, so it runs with no GPU and no
Detectron2 install. The visual language mirrors the HTML study
playgrounds: magenta/blue accents, nadir vehicles, small people from
altitude, cluttered backgrounds.

Categories (COCO ids are 1-indexed):
    1 = car     2 = person     3 = tree

Key entry points:
    make_scene(seed, kind)          -> (PIL.Image, [ann, ...])
    build_coco(n, out_dir, split)   -> path to COCO json (+ images on disk)
    THING_CLASSES                   -> ["car", "person", "tree"]
"""
import json, math, os, random
import numpy as np
from PIL import Image, ImageDraw

THING_CLASSES = ["car", "person", "tree"]
W_DEF, H_DEF = 512, 512

# ---- palette (kept close to the HTML playgrounds) -------------------------
ASPHALT = (157, 162, 148)
FIELD   = (170, 182, 132)
CROP    = (139, 155, 113)
DIRT    = (176, 152, 122)
LANE    = (232, 228, 210)
CAUTION = (200, 150, 40)
CAR_COLORS = [(122, 48, 48), (47, 90, 138), (90, 106, 47), (90, 63, 110), (60, 60, 66)]


def _poly_from_ellipse(cx, cy, rx, ry, n=16):
    return [(cx + rx * math.cos(2 * math.pi * i / n),
             cy + ry * math.sin(2 * math.pi * i / n)) for i in range(n)]


def _rect_poly(x, y, w, h):
    return [(x, y), (x + w, y), (x + w, y + h), (x, y + h)]


def _draw_car(dr, box, color):
    x, y, w, h = box
    pad = min(w, h) * 0.06
    dr.rounded_rectangle([x + pad, y + pad, x + w - pad, y + h - pad],
                         radius=min(w, h) * 0.16, fill=color, outline=(14, 18, 22))
    vertical = h >= w
    if vertical:
        dr.rounded_rectangle([x + w * 0.22, y + h * 0.30, x + w * 0.78, y + h * 0.64],
                             radius=3, fill=(143, 160, 173))
        dr.rounded_rectangle([x + w * 0.26, y + h * 0.12, x + w * 0.74, y + h * 0.24],
                             radius=2, fill=(199, 210, 218))
    else:
        dr.rounded_rectangle([x + w * 0.30, y + h * 0.22, x + w * 0.64, y + h * 0.78],
                             radius=3, fill=(143, 160, 173))
        dr.rounded_rectangle([x + w * 0.12, y + h * 0.26, x + w * 0.24, y + h * 0.74],
                             radius=2, fill=(199, 210, 218))
    return _rect_poly(x + pad, y + pad, w - 2 * pad, h - 2 * pad)


def _draw_person(dr, box):
    x, y, w, h = box
    cx, cy = x + w / 2, y + h / 2
    r = min(w, h) * 0.30
    dr.ellipse([cx - r * 1.1, cy + r * 0.1, cx + r * 1.1, cy + r * 0.9], fill=(0, 0, 0, 60))
    dr.ellipse([cx - r * 0.85, cy + r * 0.0, cx + r * 0.85, cy + r * 1.0], fill=(107, 79, 140), outline=(14, 18, 22))
    dr.ellipse([cx - r * 0.55, cy - r * 0.8, cx + r * 0.55, cy + r * 0.3], fill=(202, 162, 122), outline=(14, 18, 22))
    return _poly_from_ellipse(cx, cy, w * 0.42, h * 0.42, 14)


def _draw_tree(dr, box):
    x, y, w, h = box
    cx, cy = x + w / 2, y + h / 2
    r = min(w, h) * 0.5
    dr.ellipse([cx - r * 0.95, cy - r * 0.9, cx + r * 0.95, cy + r * 0.95], fill=(79, 122, 67), outline=(47, 74, 40))
    dr.ellipse([cx - r * 0.6, cy - r * 0.55, cx + r * 0.2, cy + r * 0.1], fill=(95, 143, 80))
    return _poly_from_ellipse(cx, cy, r * 0.9, r * 0.9, 16)


def _backdrop(img, dr, kind, rng):
    W, H = img.size
    if kind == "field":
        dr.rectangle([0, 0, W, H], fill=FIELD)
        for yy in range(8, H, 15):
            dr.line([(0, yy), (W, yy)], fill=CROP, width=6)
        # a curved dirt track
        pts = [(0, H * 0.72)]
        for t in range(1, 11):
            pts.append((W * t / 10, H * (0.62 + 0.12 * math.sin(t))))
        dr.line(pts, fill=DIRT, width=18)
    else:
        dr.rectangle([0, 0, W, H], fill=ASPHALT)
        for x in range(40, W - 20, 52):
            dr.line([(x, 14), (x, H * 0.42)], fill=LANE, width=2)
            dr.line([(x, H * 0.58), (x, H - 14)], fill=LANE, width=2)
        for x in range(0, W, 26):  # dashed centre caution line
            dr.line([(x, H * 0.5), (x + 14, H * 0.5)], fill=CAUTION, width=2)


def _overlaps(box, placed, slack=-6):
    x, y, w, h = box
    for (px, py, pw, ph) in placed:
        if (x < px + pw - slack and x + w > px + slack and
                y < py + ph - slack and y + h > py + slack):
            return True
    return False


def make_scene(seed=0, kind=None, size=(W_DEF, H_DEF)):
    """Return (PIL.Image RGB, annotations). Each annotation:
       {category_id, bbox:[x,y,w,h], segmentation:[[...]], area, iscrowd}."""
    rng = random.Random(seed)
    if kind is None:
        kind = rng.choice(["lot", "field"])
    W, H = size
    img = Image.new("RGB", (W, H))
    dr = ImageDraw.Draw(img, "RGBA")
    _backdrop(img, dr, kind, rng)

    anns, placed = [], []
    n_cars = rng.randint(3, 6)
    n_people = rng.randint(2, 5)
    n_trees = rng.randint(1, 3)

    def place(cat, wr, hr, tries=40):
        for _ in range(tries):
            w = rng.uniform(*wr); h = rng.uniform(*hr)
            if rng.random() < 0.5 and cat == 1:  # some cars rotated to horizontal
                w, h = h, w
            x = rng.uniform(6, W - w - 6); y = rng.uniform(6, H - h - 6)
            box = (x, y, w, h)
            if not _overlaps(box, placed):
                placed.append(box); return box
        return None

    for _ in range(n_cars):
        b = place(1, (60, 96), (86, 120))
        if b:
            poly = _draw_car(dr, b, rng.choice(CAR_COLORS))
            anns.append(_ann(1, b, poly))
    for _ in range(n_people):  # small from altitude
        b = place(2, (14, 26), (18, 34))
        if b:
            poly = _draw_person(dr, b)
            anns.append(_ann(2, b, poly))
    for _ in range(n_trees):
        b = place(3, (54, 90), (54, 90))
        if b:
            poly = _draw_tree(dr, b)
            anns.append(_ann(3, b, poly))
    return img, anns


def _ann(cat, box, poly):
    x, y, w, h = box
    seg = [round(float(v), 1) for pt in poly for v in pt]
    return {"category_id": cat, "bbox": [round(float(x), 1), round(float(y), 1),
            round(float(w), 1), round(float(h), 1)],
            "segmentation": [seg], "area": float(w * h), "iscrowd": 0}


def build_coco(n=40, out_dir="drone_coco", split="train", size=(W_DEF, H_DEF), seed0=0):
    """Write n images + one COCO json. Returns (json_path, img_dir)."""
    img_dir = os.path.join(out_dir, split)
    os.makedirs(img_dir, exist_ok=True)
    images, annotations = [], []
    ann_id = 1
    for i in range(n):
        img, anns = make_scene(seed=seed0 + i, size=size)
        fn = f"{split}_{i:04d}.png"
        img.save(os.path.join(img_dir, fn))
        images.append({"id": i, "file_name": fn, "width": size[0], "height": size[1]})
        for a in anns:
            a = dict(a); a["id"] = ann_id; a["image_id"] = i
            annotations.append(a); ann_id += 1
    coco = {"images": images, "annotations": annotations,
            "categories": [{"id": i + 1, "name": c} for i, c in enumerate(THING_CLASSES)]}
    jp = os.path.join(out_dir, f"{split}.json")
    with open(jp, "w") as f:
        json.dump(coco, f)
    return jp, img_dir


# convenience: numpy image + boxes for the metric notebooks
def scene_arrays(seed=0, kind="lot", size=(W_DEF, H_DEF)):
    img, anns = make_scene(seed, kind, size)
    boxes = np.array([a["bbox"] for a in anns], dtype=float)  # xywh
    labels = np.array([a["category_id"] for a in anns], dtype=int)
    return np.asarray(img), boxes, labels


In [ ]:
import importlib, drone_synth; importlib.reload(drone_synth)
from drone_synth import make_scene, build_coco, THING_CLASSES, scene_arrays
print('classes:', THING_CLASSES)

### Preview a few scenes

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1,3, figsize=(15,5))
for ax,(s,k) in zip(axs, [(3,'lot'),(7,'field'),(11,'lot')]):
    img,_ = make_scene(seed=s, kind=k)
    ax.imshow(img); ax.set_title(f'seed={s} · {k}'); ax.axis('off')
plt.tight_layout(); plt.show()

### Build a COCO dataset on disk

In [ ]:
train_json, train_dir = build_coco(n=40, out_dir='drone_coco', split='train', seed0=0)
val_json,   val_dir   = build_coco(n=12, out_dir='drone_coco', split='val',   seed0=1000)
print('train json:', train_json); print('val json:', val_json)

### Dataset statistics — note how small the *person* boxes are

In [ ]:
import json, numpy as np
coco = json.load(open(train_json))
from collections import Counter
cnt = Counter(a['category_id'] for a in coco['annotations'])
print('instances per class:', {THING_CLASSES[k-1]:v for k,v in sorted(cnt.items())})
areas = {c:[] for c in THING_CLASSES}
for a in coco['annotations']:
    areas[THING_CLASSES[a['category_id']-1]].append(a['bbox'][2]*a['bbox'][3])
for c,v in areas.items():
    print(f'{c:7s} median box area ≈ {int(np.median(v)):6d}px²  (min {int(min(v))})')

### Register with Detectron2 (optional)
`register_coco_instances` is all Detectron2 needs to consume the data. Skip if you have not installed Detectron2 — the numpy notebooks don't require it.

In [ ]:
# --- OPTIONAL: install Detectron2 to run the "library equivalent" cells ---
# This notebook's teaching content runs fully on numpy/Pillow WITHOUT this.
# Run this only if you also want the Detectron2 API demonstrations.
!python -m pip install -q 'pyyaml==6.0.*'
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
try:
    import detectron2; print("detectron2", detectron2.__version__, "ready")
except Exception as e:
    print("Detectron2 not available; the numpy cells still work.\n", e)


In [ ]:
try:
    from detectron2.data.datasets import register_coco_instances
    from detectron2.data import MetadataCatalog, DatasetCatalog
    for n in ['drone_train','drone_val']:
        if n in DatasetCatalog.list(): DatasetCatalog.remove(n)
    register_coco_instances('drone_train', {}, train_json, train_dir)
    register_coco_instances('drone_val',   {}, val_json,   val_dir)
    print('registered:', DatasetCatalog.list())
    from detectron2.utils.visualizer import Visualizer
    import random, numpy as np
    d = random.choice(DatasetCatalog.get('drone_train'))
    import cv2; im = cv2.imread(d['file_name'])[:,:,::-1]
    v = Visualizer(im, MetadataCatalog.get('drone_train'), scale=1.0)
    out = v.draw_dataset_dict(d)
    plt.figure(figsize=(6,6)); plt.imshow(out.get_image()); plt.axis('off'); plt.show()
except Exception as e:
    print('Detectron2 viz skipped:', e)